# Visualización: Ingresos Brutos por Producto y Estación del Año
### Región Metropolitana de Santiago — Mercado Mayorista (2022–2026)

---

**Pregunta de investigación:**  
*¿Qué productos de mayor volumen generan más ingresos brutos por época del año?*


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Paleta de colores por estación (hemisferio sur)
SEASON_COLORS = {
    'Verano':    '#F4A261',   # naranja cálido
    'Otoño':     '#E76F51',   # terracota
    'Invierno':  '#457B9D',   # azul frío
    'Primavera': '#52B788',   # verde fresco
}
SEASON_ORDER = ['Verano', 'Otoño', 'Invierno', 'Primavera']

print('Librerías cargadas ✓')

### 1. Carga y preparación de datos
Se reutiliza el `frutas_rm` ya procesado con las columnas `Month`, `Season` e `Ingreso Bruto`.

In [ ]:
fruta_2022 = pd.read_csv('precio_mayorista_fruta-hortaliza_2022.csv', decimal=',')
fruta_2023 = pd.read_csv('precio_mayorista_fruta-hortaliza_2023.csv', decimal=',')
fruta_2024 = pd.read_csv('precio_mayorista_fruta-hortaliza_2024.csv', decimal=',')
fruta_2025 = pd.read_csv('precio_mayorista_fruta-hortaliza_2025.csv', decimal=',')
fruta_2026 = pd.read_csv('precio_mayorista_fruta-hortaliza_2026.csv', decimal=',')

fruta_data = pd.concat([
    fruta_2022.dropna(how='all'),
    fruta_2023.dropna(how='all'),
    fruta_2024.dropna(how='all'),
    fruta_2025.dropna(how='all'),
    fruta_2026.dropna(how='all'),
], ignore_index=True).loc[:, [
    'Fecha','ID region','Region','Mercado','Subsector',
    'Producto','Variedad / Tipo','Calidad','Unidad de comercializacion',
    'Origen','Volumen','Precio minimo','Precio maximo','Precio promedio'
]]

frutas_rm = fruta_data[fruta_data['Region'] == 'Región Metropolitana de Santiago'].copy()

# ── Columnas derivadas ──────────────────────────────────────────────────────
frutas_rm['Fecha'] = pd.to_datetime(frutas_rm['Fecha'])
frutas_rm['Month'] = frutas_rm['Fecha'].dt.month
mmdd = frutas_rm['Fecha'].dt.month * 100 + frutas_rm['Fecha'].dt.day

condiciones = [
    (mmdd >= 1221) | (mmdd < 320),
    (mmdd >= 320)  & (mmdd < 621),
    (mmdd >= 621)  & (mmdd <= 921),
    (mmdd >= 922)  & (mmdd < 1221),
]
frutas_rm['Season']       = np.select(condiciones, SEASON_ORDER, default='Fecha inválida')
frutas_rm['Ingreso Bruto'] = frutas_rm['Volumen'] * (frutas_rm['Precio promedio'] / 10000)

print(f'Registros RM: {len(frutas_rm):,}  |  Productos únicos: {frutas_rm["Producto"].nunique()}')
frutas_rm[['Fecha','Producto','Season','Volumen','Precio promedio','Ingreso Bruto']].head()

### 2. Agregación: Top 10 productos por Ingreso Bruto total

In [ ]:
# Top 10 productos por ingreso bruto acumulado
top10 = (
    frutas_rm.groupby('Producto')['Ingreso Bruto']
    .sum()
    .nlargest(10)
    .index.tolist()
)

# Tabla pivot: producto × estación
df_top = frutas_rm[frutas_rm['Producto'].isin(top10)]

pivot_ib = (
    df_top.groupby(['Producto','Season'])['Ingreso Bruto']
    .sum()
    .unstack('Season')
    .reindex(columns=SEASON_ORDER, fill_value=0)
)
pivot_ib['Total'] = pivot_ib.sum(axis=1)
pivot_ib = pivot_ib.sort_values('Total', ascending=False).drop(columns='Total')

pivot_vol = (
    df_top.groupby(['Producto','Season'])['Volumen']
    .sum()
    .unstack('Season')
    .reindex(columns=SEASON_ORDER, fill_value=0)
    .loc[pivot_ib.index]   # mismo orden
)

print('Pivot Ingreso Bruto (top 10 productos × estación):')
pivot_ib.style.format('{:,.0f}').background_gradient(cmap='YlOrRd', axis=None)

---
## Gráfico 1 — Barras apiladas: Ingreso Bruto por Estación

### ¿Por qué este tipo de gráfico?

Se eligió un **gráfico de barras apiladas horizontales** porque permite responder **dos preguntas al mismo tiempo**:

1. **¿Cuánto ingreso bruto total genera cada producto?** → lo dice la longitud total de la barra.
2. **¿En qué estación concentra ese ingreso?** → lo revela la proporción de cada segmento de color.

La orientación **horizontal** es más efectiva cuando los nombres de productos son largos, ya que evita rotar etiquetas y mantiene la legibilidad. Ordenar las barras de mayor a menor ingreso total permite una **comparación jerárquica inmediata** sin necesidad de leer los números.

> **Mensaje central:** *Tomate, Papa y Cebolla dominan el ingreso bruto; y en todos ellos la distribución por estación revela patrones de estacionalidad relevantes para la planificación comercial.*

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

y = np.arange(len(pivot_ib))
left = np.zeros(len(pivot_ib))

for season in SEASON_ORDER:
    valores = pivot_ib[season].values
    bars = ax.barh(y, valores, left=left,
                   color=SEASON_COLORS[season], label=season,
                   edgecolor='white', linewidth=0.5)
    # Etiqueta dentro del segmento si es suficientemente ancho
    for rect, val in zip(bars, valores):
        w = rect.get_width()
        if w > pivot_ib.values.sum(axis=1).max() * 0.05:
            ax.text(rect.get_x() + w / 2, rect.get_y() + rect.get_height() / 2,
                    f'{val:,.0f}', ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold')
    left += valores

ax.set_yticks(y)
ax.set_yticklabels(pivot_ib.index, fontsize=10)
ax.set_xlabel('Ingreso Bruto Estimado (Volumen × Precio Promedio / 10,000)', fontsize=9)
ax.set_title('Top 10 Productos — Ingreso Bruto por Estación del Año\nRegión Metropolitana · 2022–2026',
             fontsize=13, fontweight='bold', pad=14)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(title='Estación', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('grafico1_barras_apiladas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 1 guardado ✓')

---
## Gráfico 2 — Heatmap: Intensidad de Ingreso Bruto (Producto × Estación)

### ¿Por qué este tipo de gráfico?

El **mapa de calor (heatmap)** es ideal cuando queremos detectar **patrones de intensidad en una matriz** de dos variables categóricas: producto y estación.

Mientras las barras apiladas son buenas para comparar totales, el heatmap permite **ver de un vistazo** en qué combinación producto–estación se concentra o escasea el ingreso bruto. La gradiente de color (de blanco/amarillo a rojo oscuro) activa la percepción visual de forma intuitiva: *lo más rojo, lo más rentable*.

> **Mensaje central:** *El heatmap revela qué estaciones «encienden» a cada producto — por ejemplo, si un producto es rojo en Verano y pálido en Invierno, su estacionalidad es clara y explotable.*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

data_matrix = pivot_ib.values
# Normalizar por fila para que cada producto muestre su distribución relativa
row_sums = data_matrix.sum(axis=1, keepdims=True)
data_norm = np.where(row_sums > 0, data_matrix / row_sums * 100, 0)

im = ax.imshow(data_norm, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100)

ax.set_xticks(range(len(SEASON_ORDER)))
ax.set_xticklabels(SEASON_ORDER, fontsize=11)
ax.set_yticks(range(len(pivot_ib)))
ax.set_yticklabels(pivot_ib.index, fontsize=10)

for i in range(len(pivot_ib)):
    for j in range(len(SEASON_ORDER)):
        val = data_norm[i, j]
        color = 'white' if val > 55 else '#333333'
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                fontsize=8.5, color=color, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('% del Ingreso Bruto del Producto', fontsize=9)

ax.set_title('Distribución Relativa del Ingreso Bruto por Estación\n(% del total de cada producto) · RM 2022–2026',
             fontsize=12, fontweight='bold', pad=14)
ax.set_xlabel('Estación del Año', fontsize=10)

plt.tight_layout()
plt.savefig('grafico2_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 2 guardado ✓')

---
## Gráfico 3 — Barras agrupadas: Volumen vs. Ingreso Bruto (Top 10)

### ¿Por qué este tipo de gráfico?

Las **barras agrupadas doble-eje** permiten comparar directamente **volumen e ingreso bruto** para el mismo producto. Dado que no tenemos costo de producción, este gráfico ayuda a detectar si un producto es **eficiente en precio** (poco volumen pero mucho ingreso) versus **eficiente en cantidad** (mucho volumen pero precio bajo).

El doble eje Y es necesario porque las escalas de volumen y de ingreso son distintas, pero querer verlos juntos es legítimo aquí porque la pregunta es precisamente esa relación.

> **Mensaje central:** *Algunos productos como Tomate generan alto ingreso por volumen moderado (precio unitario alto), mientras otros requieren grandes volúmenes para alcanzar ingresos similares.*

In [ ]:
resumen = (
    frutas_rm[frutas_rm['Producto'].isin(top10)]
    .groupby('Producto')
    .agg(Volumen_Total=('Volumen','sum'), Ingreso_Total=('Ingreso Bruto','sum'))
    .loc[pivot_ib.index]
)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

x = np.arange(len(resumen))
w = 0.38

b1 = ax1.bar(x - w/2, resumen['Volumen_Total'], width=w,
             color='#457B9D', alpha=0.85, label='Volumen Total')
b2 = ax2.bar(x + w/2, resumen['Ingreso_Total'], width=w,
             color='#F4A261', alpha=0.85, label='Ingreso Bruto Total')

ax1.set_xticks(x)
ax1.set_xticklabels(resumen.index, rotation=20, ha='right', fontsize=10)
ax1.set_ylabel('Volumen Total (unidades/kilos/etc.)', color='#457B9D', fontsize=9)
ax2.set_ylabel('Ingreso Bruto Estimado', color='#F4A261', fontsize=9)
ax1.tick_params(axis='y', labelcolor='#457B9D')
ax2.tick_params(axis='y', labelcolor='#F4A261')

ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

lines = [mpatches.Patch(color='#457B9D', label='Volumen Total'),
         mpatches.Patch(color='#F4A261', label='Ingreso Bruto Total')]
ax1.legend(handles=lines, loc='upper right', fontsize=9)

ax1.set_title('Top 10 Productos — Volumen vs. Ingreso Bruto Acumulado (2022–2026)\nRegión Metropolitana',
              fontsize=12, fontweight='bold', pad=12)
ax1.spines[['top']].set_visible(False)
ax2.spines[['top']].set_visible(False)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('grafico3_volumen_vs_ingreso.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 3 guardado ✓')

---
## Gráfico 4 — Línea temporal: Ingreso Bruto mensual (Top 5 productos)

### ¿Por qué este tipo de gráfico?

El **gráfico de líneas temporales** es el más efectivo para mostrar **tendencias y estacionalidad a lo largo del tiempo**. Al usar el mes como eje X y superponer los 5 productos con mayor ingreso, se puede leer directamente:

- Los **picos estacionales** de cada producto (¿en qué mes generan más ingreso?).
- Si los productos compiten por el mismo periodo de alta demanda o se **complementan** a lo largo del año.
- La **consistencia o volatilidad** del ingreso mes a mes.

Las bandas de fondo identifican las estaciones del año para que la lectura sea inmediata sin necesidad de conocer el calendario astronómico del hemisferio sur.

> **Mensaje central:** *Las líneas revelan qué productos son «de temporada» (pico pronunciado en pocos meses) versus qué productos tienen demanda constante durante el año.*

In [ ]:
top5 = pivot_ib.sum(axis=1).nlargest(5).index.tolist()

monthly = (
    frutas_rm[frutas_rm['Producto'].isin(top5)]
    .groupby(['Producto','Month'])['Ingreso Bruto']
    .sum()
    .unstack('Producto')
    .reindex(range(1, 13), fill_value=0)
)

MONTH_LABELS = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']

# Bandas de estación (hemisferio sur)
# Verano: dic(12)–mar(3) => franjas al inicio y fin
SEASON_BANDS = [
    (1,  2,  'Verano',    '#F4A261'),
    (3,  5,  'Otoño',     '#E76F51'),
    (6,  8,  'Invierno',  '#457B9D'),
    (9,  11, 'Primavera', '#52B788'),
    (12, 12, 'Verano',    '#F4A261'),
]

LINE_COLORS = ['#264653','#2A9D8F','#E9C46A','#F4A261','#E76F51']

fig, ax = plt.subplots(figsize=(13, 6))

# Bandas de fondo
for m_start, m_end, label, color in SEASON_BANDS:
    ax.axvspan(m_start - 0.5, m_end + 0.5, alpha=0.10, color=color)
    if m_start != 12:   # no duplicar etiqueta de Verano
        ax.text((m_start + m_end) / 2, ax.get_ylim()[1],
                label, ha='center', va='bottom',
                fontsize=8, color=color, fontstyle='italic')

for i, prod in enumerate(top5):
    ax.plot(monthly.index, monthly[prod],
            marker='o', markersize=4,
            color=LINE_COLORS[i], linewidth=2,
            label=prod)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_LABELS, fontsize=10)
ax.set_ylabel('Ingreso Bruto Estimado (acumulado 2022–2026)', fontsize=9)
ax.set_title('Ingreso Bruto Mensual — Top 5 Productos · Región Metropolitana 2022–2026',
             fontsize=12, fontweight='bold', pad=14)
ax.legend(title='Producto', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('grafico4_lineas_temporales.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 4 guardado ✓')

---
## Resumen de decisiones de visualización

| Gráfico | Tipo | ¿Por qué este formato? |
|---------|------|------------------------|
| **1** | Barras apiladas horizontales | Compara totales Y composición por estación en una sola lectura. Horizontal por nombres largos. |
| **2** | Heatmap (matriz de calor) | Detecta patrones de intensidad en la combinación producto×estación. Lectura visual inmediata de concentración. |
| **3** | Barras agrupadas doble eje | Pone en perspectiva si el ingreso viene del precio o del volumen, sin costo de producción disponible. |
| **4** | Líneas temporales con bandas | Muestra tendencias mes a mes y estacionalidad real. Las bandas contextualizan hemisferio sur sin texto adicional. |

### Limitación importante
El **Ingreso Bruto** aquí calculado es `Volumen × Precio Promedio / 10,000`. **No es ganancia neta**, ya que no contamos con el costo de producción o adquisición. Sin embargo, sirve como *proxy* para identificar qué productos y qué épocas del año son más relevantes comercialmente dentro del mercado mayorista de la Región Metropolitana.
